# **Checkpoint 2 - Dynamic Programming**

---

**Integrantes:**

Artur Alves Tenca - RM555171

Igor Brunelli Ralo - RM555035



---



# Desafio 1 - Metro de Londres

O desafio pede uma entrada com a estação de entrada e saída, horario de entrada e escolher entre caminho curto, medio e longo. Usamos os requerimentos da programação dinâmica para realizar a entrega, como list comprenhention e grafos. No final utilizamos Folium para exibir a rota no mapa de Londres.

In [ ]:
!pip install haversine

In [2]:
import re
import json
import folium
import pandas as pd
from collections import deque
from haversine import haversine
from functools import lru_cache
from IPython.display import display

Carrega os dados do CSV com estações e JSON com conexões do metrô

In [3]:
# Carregamento de dados
pd.set_option('display.max_rows', None)
df = pd.read_csv('LondonStations.csv')
with open('stations.json') as f:
    metro_data = json.load(f)

# Normaliza nomes no estilo do JSON
def normalizar_nome(nome):
    nome = re.sub(r"\b(STATION|UNDERGROUND)\b", "", nome.upper())
    return re.sub(r"\s+", " ", nome.replace("'", "")).strip()

df["Station"] = df["Station"].apply(normalizar_nome)

Cria um grafo onde cada estação aponta para suas vizinhas conectadas diretamente

In [4]:
# Grafo de conexões
graph = {
    station: {conn["station"] for line in info["lines"].values() for conn in line["stations"]}
    for station, info in metro_data.items()
}

# Linhas por estação
estacoes_linhas = {
    station: list(info.get("lines", {}).keys())
    for station, info in metro_data.items()
}

Função memoizada para retornar latitude e longitude de uma estação, evitando cálculo repetido

In [9]:
# Coordenadas com memoization
@lru_cache(maxsize=None)
def LatLong(estacao):
    linha = df[df["Station"] == estacao]
    return tuple(linha.iloc[0][["Latitude", "Longitude"]]) if not linha.empty else None

Encontra o caminho com menor número de estações entre duas estações usando busca em largura. Tambêm busca em profundidade para encontrar o maior caminho possível. Por fim varre vários caminhos curtos e retorna o caminho de comprimento médio entre origem e destino


In [10]:
# Cálculo da rota mais rapida
def encontrar_rota_menor(origem, destino):
    queue, visited = deque([(origem, [origem])]), set()
    while queue:
        atual, caminho = queue.popleft()
        if atual == destino:
            return caminho
        visited.add(atual)
        queue.extend((viz, caminho + [viz]) for viz in graph.get(atual, []) if viz not in visited)
    return []


# Cálculo da rota média
def encontrar_rota_media(origem, destino):
    visited = set()
    queue = deque([(origem, [origem])])
    melhores_caminhos = []
    while queue:
        atual, caminho = queue.popleft()
        if atual == destino:
            melhores_caminhos.append(caminho)
            if len(melhores_caminhos) >= 5:
                break
        visited.add(atual)
        for viz in graph.get(atual, []):
            if viz not in caminho:
                queue.append((viz, caminho + [viz]))
    if melhores_caminhos:
        return sorted(melhores_caminhos, key=len)[len(melhores_caminhos)//2]
    return []


# Cálculo da rota mais longa
def encontrar_rota_maior(origem, destino, limite=50):
    max_path = []
    def dfs(atual, visitados, caminho):
        nonlocal max_path
        if len(caminho) > limite: return
        visitados.add(atual)
        caminho.append(atual)
        if atual == destino and len(caminho) > len(max_path):
            max_path = caminho[:]
        else:
            for viz in graph.get(atual, []):
                if viz not in visitados:
                    dfs(viz, visitados, caminho)
        caminho.pop()
        visitados.remove(atual)
    dfs(origem, set(), [])
    return max_pat

Calcula o tempo estimado de uma rota considerando distância e horário (espera) e extrai as linhas de cada estação e, caso mude de linha, mostra quantas vezes mudou.

In [6]:
# Tempo de viagem
def tempo_viagem(lista, hora):
    espera = 1.5 if hora < 11 else 2 if hora > 18 else 1
    distancias = [haversine(LatLong(lista[i]), LatLong(lista[i+1])) for i in range(len(lista)-1)]
    tempo_total = sum(d/35 for d in distancias)
    km_total = sum(distancias)
    return tempo_total*60 + espera, km_total

# Linhas e trocas
def linhas_e_trocas(lista):
    linhas = [estacoes_linhas.get(est, ["Desconhecida"])[0] for est in lista[:-1]]
    trocas = sum(1 for i in range(1, len(linhas)) if linhas[i] != linhas[i-1])
    return linhas, trocas

Plota a rota com o mapa formatado de maneira organizada e bonita

In [7]:
# Mapa
def gerar_mapa(lista):
    coords = [LatLong(est) for est in lista if LatLong(est)]
    mapa = folium.Map(location=coords[0], zoom_start=12)
    for i, (lat, lon) in enumerate(coords):
        cor, icone = ('green', 'play') if i == 0 else ('red', 'flag') if i == len(coords)-1 else ('blue', 'circle')
        folium.Marker((lat, lon), tooltip=lista[i].title(), icon=folium.Icon(color=cor, icon=icone, prefix='fa')).add_to(mapa)
    folium.PolyLine(coords, color="blue", weight=5).add_to(mapa)
    display(mapa)

Solicita dados do usuário, executa a busca e mostra tempo, linha, trocas e mapa

In [ ]:
# Entrada principal
def rota():
    inicio = normalizar_nome(input("estação de origem: "))
    fim = normalizar_nome(input("Estação de destino: "))
    hora = int(input("Hora (HH): "))
    modo = input("Modo (1-curto | 2-médio | 3-longo): ")

    if modo == '1':
        rota = encontrar_rota_menor(inicio, fim)
    elif modo == '2':
        rota = encontrar_rota_media(inicio, fim)
    elif modo == '3':
        rota = encontrar_rota_maior(inicio, fim)
    else:
        print("Modo inválido.")
        return

    if not rota:
        return print("❌ Rota não encontrada")

    tempo, km = tempo_viagem(rota, hora)
    linhas, trocas = linhas_e_trocas(rota)
    print('-' * 50)
    print(f"\nTempo estimado: {tempo:.1f} min | Distância: {km:.2f} km")
    print("Caminho:", " → ".join(est.title() for est in rota))
    print("\nLinhas:", " → ".join(linhas))
    print(f"\nTrocas de linha: {trocas}")
    print('-' * 50)
    gerar_mapa(rota)


rota()